# SERPENS Black Hole Test
Test notebook for GR-enabled simulation using the `StellarBH-10` system.

In [3]:
import os
WORKDIR = '/Users/raghavchari/SERPENS'
if os.getcwd() != WORKDIR:
    os.chdir(WORKDIR)
print('Working Directory:', os.getcwd())

Working Directory: /Users/raghavchari/SERPENS


In [4]:
from src.serpens_simulation import SerpensSimulation
from src.parameters import GLOBAL_PARAMETERS
from src.species import Species
import numpy as np

## 1. Enable GR and create the simulation
Set `gr_enabled=True` and point `gr_source` at the black hole before initializing.

In [6]:
# Enable GR corrections
GLOBAL_PARAMETERS.set('gr_enabled', True)
GLOBAL_PARAMETERS.set('gr_source', 'bh')

# Create simulation with 10 solar mass stellar black hole system
sim = SerpensSimulation(system='StellarBH-10')

FileNotFoundError: [Errno 2] Unable to synchronously create file (unable to open file: name = 'simdata/particle_params.h5', errno = 2, error message = 'No such file or directory', flags = 13, o_flags = 602)

## 2. Inspect the system
Print out bodies, the Schwarzschild radius, and ISCO.

In [ ]:
print(f'Number of bodies: {sim.N}')
print(f'N_active (gravitating): {sim.N_active}\n')

for i in range(sim.N):
    p = sim.particles[i]
    print(f'Particle {i}: m={p.m:.3e} kg, r={p.r:.3e} m, pos=({p.x:.3e}, {p.y:.3e}, {p.z:.3e})')

print(f'\nr_schwarzschild = {sim._r_schwarzschild:.3e} m')
print(f'r_isco          = {sim._r_isco:.3e} m')
print(f'r_isco / r_s    = {sim._r_isco / sim._r_schwarzschild:.1f}')

## 3. Define a species and source
Use Hydrogen as a test particle species sputtered from the disk source.

In [ ]:
H = Species(
    name='H',
    n_th=0,
    n_sp=40,
    mass_per_sec=1000,
    beta=0.0,
    lifetime=1e6,
    sput_spec={
        'model_smyth_v_b': 2000,
        'model_smyth_v_M': 30000
    }
)
print(H)

In [ ]:
sim.object_to_source('disk_source', H)

## 4. Run a short test simulation
Advance for a few spawning steps to verify GR integration and ISCO removal work.

In [ ]:
# Run 5 spawn steps over ~1 hour of simulation time
sim.advance(hours=1, spawns=5, verbose=True)

## 5. Plot particle positions
Quick scatter plot to visualize the particle cloud around the black hole.

In [ ]:
import matplotlib.pyplot as plt

# Collect test particle positions
x = [sim.particles[i].x for i in range(sim.N_active, sim.N)]
y = [sim.particles[i].y for i in range(sim.N_active, sim.N)]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(x, y, s=1, alpha=0.5, label='Test particles')

# Mark the black hole and ISCO
bh = sim.particles['bh']
isco_circle = plt.Circle((bh.x, bh.y), sim._r_isco, fill=False, color='red', linestyle='--', label='ISCO')
ax.add_patch(isco_circle)
ax.plot(bh.x, bh.y, 'ko', markersize=8, label='Black Hole')

# Mark the disk source
ds = sim.particles['disk_source']
ax.plot(ds.x, ds.y, 'g^', markersize=8, label='Disk Source')

ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_aspect('equal')
ax.legend()
ax.set_title('SERPENS Black Hole Test — Particle Positions')
plt.tight_layout()
plt.show()

print(f'Total test particles remaining: {sim.N - sim.N_active}')

## 6. SerpensAnalyzer Plots
Use the same analysis tools as the original SERPENS plotting notebook.

In [ ]:
from src.serpens_analyzer import SerpensAnalyzer
import matplotlib

# Create analyzer with disk_source as the reference system
sa = SerpensAnalyzer(reference_system="disk_source")

### Planar View (top-down orbital plane)

In [ ]:
sa.plot_planar(timestep=-1, figsize=8)

In [ ]:
# Planar plot with density scatter and custom colormap
sa.plot_planar(
    timestep=-1,
    scatter=True,
    triplot=True,
    trialpha=0.4,
    figsize=8,
    colormap=matplotlib.colormaps["magma"],
    single_plot=True,
    interactive=False
)

### Line of Sight View

In [ ]:
sa.plot_lineofsight(timestep=-1, figsize=8)

In [ ]:
# Line of sight with custom colormap
sa.plot_lineofsight(
    timestep=-1,
    scatter=True,
    figsize=8,
    colormap=matplotlib.colormaps["afmhot"],
    single_plot=True,
    interactive=False
)

### 1D Density Cut

In [ ]:
sa.plot_1d_cut(timestep=-1, figsize=(4, 4), max_distance_rp=10)

In [ ]:
# 1D cut with log scale
sa.plot_1d_cut(
    timestep=-1,
    species_num=1,
    log_scale=True,
    max_distance_rp=4,
    figsize=(4, 4)
)

### 3D Interactive Plot

In [ ]:
fig = sa.plot_3d(timestep=-1)
fig.update_layout(template="plotly_dark", title="SERPENS Black Hole — 3D Particle Distribution")
fig.show()

### Phase Curve

In [ ]:
sa.calculate_phasecurve('disk_source', orbits=1)

In [ ]:
SerpensAnalyzer.plot_phasecurve(
    filename='phase-curve.csv',
    column_density=True,
    particle_density=True,
    type="max"
)